# Qwen3-VL-4B-Instruct on SageMaker with Neuron (BYOC)

Deploy [Qwen3-VL-4B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct) to a SageMaker real-time endpoint on AWS Inferentia2 using a custom BYOC (Bring Your Own Container) with vLLM-neuron.

## Architecture

- **Container**: Custom BYOC based on `pytorch-inference-neuronx` (SDK 2.28) + vLLM 0.13.0 + vllm_neuron 0.4.1
- **Engine**: vLLM with NxD Inference (neuronx-distributed-inference) backend
- **Instance**: `ml.inf2.8xlarge` (2 NeuronCores, tp=2)
- **Host AMI**: `al2-ami-sagemaker-inference-neuron-2` (Neuron driver 2.19, required for ExtISA support)
- **Features**: Text-only chat, image+text VLM queries, OpenAI-compatible API

## Prerequisites

1. **BYOC container built and pushed to ECR** (see Step 1 for build instructions)
2. **Model weights uploaded to S3** (see Step 2)
3. **AWS permissions**: SageMaker, ECR, S3, IAM roles
4. **boto3 >= 1.42** (required for `InferenceAmiVersion` parameter)

## Performance

| Workload | Throughput | Latency | Instance |
|----------|-----------|---------|----------|
| Text-only | ~30 tok/s | 0.15-0.7s | ml.inf2.8xlarge |
| Image+Text | ~20-32 tok/s | 2.5-3.5s | ml.inf2.8xlarge |

## Why This Approach?

**Why BYOC**: No standard SageMaker DLC combines vLLM + NxD Inference + SDK 2.28. The DJL Neuron images use deprecated transformers-neuronx (SDK 2.20), and the HuggingFace vLLM NeuronX image is at SDK 2.26 / vLLM 0.11 (too old for Qwen3-VL).

**Why SDK 2.28**: SageMaker currently only offers inf2 and trn1 Neuron instances (no trn2). SDK 2.29 (NxD Inference 0.9) drops inf2/trn1 support, so SDK 2.28 is the latest compatible version.

**Why `InferenceAmiVersion`**: SageMaker's default Neuron host driver (v2.10) cannot load NEFFs with Extended ISA (ExtISA) instructions generated by the SDK 2.28 NxD Inference compiler. The `al2-ami-sagemaker-inference-neuron-2` AMI provides Neuron driver v2.19, which supports ExtISA.

## Step 1: Setup and Configuration

In [1]:
import warnings
warnings.filterwarnings('ignore')

import boto3
import json
import base64
import time
import os

sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']
region = boto3.Session().region_name
sm_client = boto3.client('sagemaker', region_name=region)
sm_runtime = boto3.client('sagemaker-runtime', region_name=region)

print(f"AWS Account: {account_id}")
print(f"Region: {region}")
print(f"boto3 version: {boto3.__version__}")

# Verify boto3 version supports InferenceAmiVersion
assert tuple(int(x) for x in boto3.__version__.split('.')[:2]) >= (1, 42), \
    f"boto3 >= 1.42 required for InferenceAmiVersion. Got {boto3.__version__}. Run: pip install --upgrade boto3"

AWS Account: 691320336001
Region: us-west-2
boto3 version: 1.42.97

In [2]:
# ============================================================
# USER CONFIGURATION - Edit these values
# ============================================================

# SageMaker execution role
ROLE_ARN = "arn:aws:iam::691320336001:role/service-role/AmazonSageMaker-ExecutionRole-20250910T103661"

# ECR container (must be pre-built -- see build instructions below)
ECR_REPO = "qwen3vl-neuron-byoc"
ECR_TAG = "v9b"
IMAGE_URI = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ECR_REPO}:{ECR_TAG}"

# S3 path to model weights (HuggingFace format, not tar.gz)
# Must contain: config.json, tokenizer files, safetensors weights
S3_BUCKET = f"sagemaker-{region}-{account_id}"
S3_MODEL_PREFIX = "qwen3vl-neuron/model-weights-only"
MODEL_DATA_URI = f"s3://{S3_BUCKET}/{S3_MODEL_PREFIX}/"

# Instance configuration
INSTANCE_TYPE = "ml.inf2.8xlarge"  # 2 NeuronCores, tp=2

# CRITICAL: Use the updated Neuron host AMI with driver v2.19
# Without this, the default driver (v2.10) cannot load ExtISA NEFFs
INFERENCE_AMI_VERSION = "al2-ami-sagemaker-inference-neuron-2"

print(f"Container: {IMAGE_URI}")
print(f"Instance: {INSTANCE_TYPE}")
print(f"Model data: {MODEL_DATA_URI}")
print(f"Host AMI: {INFERENCE_AMI_VERSION}")

Container: 691320336001.dkr.ecr.us-west-2.amazonaws.com/qwen3vl-neuron-byoc:v9b
Instance: ml.inf2.8xlarge
Model data: s3://sagemaker-us-west-2-691320336001/qwen3vl-neuron/model-weights-only/
Host AMI: al2-ami-sagemaker-inference-neuron-2

## Step 2: Upload Model Weights to S3

Download the model from HuggingFace and upload to S3. Skip this cell if weights are already in S3.

The model weights are ~8.3 GB (2 safetensors shards + tokenizer + config files).

In [3]:
# Check if model weights already exist in S3
s3_client = boto3.client('s3')
try:
    resp = s3_client.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_MODEL_PREFIX, MaxKeys=5)
    count = resp.get('KeyCount', 0)
    if count > 0:
        print(f"Model weights already in S3 ({count}+ files). Skipping download.")
        print(f"  s3://{S3_BUCKET}/{S3_MODEL_PREFIX}/")
    else:
        print("No model weights found in S3. Run the next cell to download and upload.")
except Exception as e:
    print(f"Error checking S3: {e}")
    print("Run the next cell to download and upload model weights.")

Model weights already in S3 (5+ files). Skipping download.
  s3://sagemaker-us-west-2-691320336001/qwen3vl-neuron/model-weights-only/

In [ ]:
# Download from HuggingFace and upload to S3 (skip if already uploaded)
# This cell requires: pip install huggingface_hub

# from huggingface_hub import snapshot_download
# 
# MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
# LOCAL_PATH = "/tmp/qwen3vl_model"
# 
# print(f"Downloading {MODEL_ID}...")
# snapshot_download(
#     repo_id=MODEL_ID,
#     local_dir=LOCAL_PATH,
#     ignore_patterns=["*.gguf", "*.md", ".gitattributes"]
# )
# 
# # Upload to S3
# import subprocess
# cmd = f"aws s3 sync {LOCAL_PATH} s3://{S3_BUCKET}/{S3_MODEL_PREFIX}/ --exclude '*.md' --exclude '.git*'"
# print(f"Uploading to S3...")
# subprocess.run(cmd, shell=True, check=True)
# print("Upload complete.")

## Step 3: Build and Push BYOC Container

The container must be built on a Docker-capable machine (not this notebook instance).

### Container Architecture

The BYOC container is built from the official `pytorch-inference-neuronx:2.9.0-neuronx-py312-sdk2.28.0-ubuntu24.04` DLC with:

1. **vLLM 0.13.0** installed from PyPI (for dependencies), then replaced with the DLAMI's exact vLLM package
2. **vllm_neuron 0.4.1** plugin extracted from the Neuron DLAMI
3. **Sampler import patch** for vllm_neuron (fixes TypeError when on-device sampling is disabled)
4. **FastAPI serving code** (model.py) with health check at `/ping` and inference at `/invocations`

### Build Steps

```bash
# On a Docker-capable machine (e.g., m5.large)

# 1. Login to ECR (DLC base image)
aws ecr get-login-password --region $REGION | \
  docker login --username AWS --password-stdin 763104351884.dkr.ecr.$REGION.amazonaws.com

# 2. Login to your ECR repo
aws ecr get-login-password --region $REGION | \
  docker login --username AWS --password-stdin $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com

# 3. Build (artifacts/ must contain vllm-dlami-package.tar.gz and vllm-neuron-code.tar.gz)
docker build --build-arg REGION=$REGION -t qwen3vl-neuron-byoc:v9b .

# 4. Push
docker tag qwen3vl-neuron-byoc:v9b $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com/qwen3vl-neuron-byoc:v9b
docker push $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com/qwen3vl-neuron-byoc:v9b
```

The DLAMI artifacts (`vllm-dlami-package.tar.gz`, `vllm-neuron-code.tar.gz`) are extracted from a running Neuron DLAMI instance (SDK 2.28, `Deep Learning AMI Neuron (Ubuntu 24.04) 20260227`).

In [4]:
# Verify container exists in ECR
ecr_client = boto3.client('ecr', region_name=region)
try:
    resp = ecr_client.describe_images(
        repositoryName=ECR_REPO,
        imageIds=[{'imageTag': ECR_TAG}]
    )
    img = resp['imageDetails'][0]
    size_mb = img.get('imageSizeInBytes', 0) / 1024 / 1024
    pushed = img.get('imagePushedAt', 'unknown')
    print(f"Container found: {IMAGE_URI}")
    print(f"  Size: {size_mb:.0f} MB")
    print(f"  Pushed: {pushed}")
except Exception as e:
    print(f"Container NOT found: {IMAGE_URI}")
    print(f"Error: {e}")
    print("Build and push the container first (see instructions above).")

Container found: 691320336001.dkr.ecr.us-west-2.amazonaws.com/qwen3vl-neuron-byoc:v9b
  Size: 10414 MB
  Pushed: 2026-04-17 00:51:27.675000-04:00

## Step 4: Deploy SageMaker Endpoint

**Startup time**: ~15 minutes on first deployment (model download + Neuron compilation from scratch).

We use boto3 directly (instead of the SageMaker SDK) because the `InferenceAmiVersion` parameter is required to get Neuron driver v2.19, and the SageMaker SDK's `Model.deploy()` does not expose this parameter.

### Why `InferenceAmiVersion` is Required

The NxD Inference compiler (SDK 2.28) generates NEFFs containing Extended ISA (ExtISA) instructions via NKI kernel calls. SageMaker's default Neuron host driver (v2.10) cannot execute these instructions, causing `Allocation Failure` errors at model load time. The `al2-ami-sagemaker-inference-neuron-2` AMI provides driver v2.19, which supports ExtISA.

This affects ALL models deployed via vLLM + vllm_neuron + NxD Inference on SageMaker.

In [5]:
timestamp = int(time.time())
model_name = f"qwen3vl-neuron-{timestamp}"
config_name = f"qwen3vl-neuron-config-{timestamp}"
endpoint_name = f"qwen3vl-neuron-{timestamp}"

# Step 4a: Create SageMaker Model
print(f"Creating model: {model_name}")
sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer={
        'Image': IMAGE_URI,
        'ModelDataSource': {
            'S3DataSource': {
                'S3Uri': MODEL_DATA_URI,
                'S3DataType': 'S3Prefix',
                'CompressionType': 'None'
            }
        }
    },
    ExecutionRoleArn=ROLE_ARN
)
print(f"  Model created.")

# Step 4b: Create Endpoint Configuration with InferenceAmiVersion
print(f"Creating endpoint config: {config_name}")
sm_client.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[{
        'VariantName': 'AllTraffic',
        'ModelName': model_name,
        'InstanceType': INSTANCE_TYPE,
        'InitialInstanceCount': 1,
        'InitialVariantWeight': 1.0,
        'ContainerStartupHealthCheckTimeoutInSeconds': 1800,  # 30 min for compile
        'ModelDataDownloadTimeoutInSeconds': 1800,             # 30 min for download
        'VolumeSizeInGB': 100,
        # CRITICAL: Use updated Neuron host AMI with driver v2.19
        'InferenceAmiVersion': INFERENCE_AMI_VERSION
    }]
)
print(f"  Endpoint config created with InferenceAmiVersion={INFERENCE_AMI_VERSION}")

# Step 4c: Create Endpoint
print(f"Creating endpoint: {endpoint_name}")
sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)
print(f"  Endpoint creation started.")
print(f"\nThis will take ~15 minutes (model download + Neuron compilation).")
print(f"Monitor progress in CloudWatch: /aws/sagemaker/Endpoints/{endpoint_name}")

Creating model: qwen3vl-neuron-v10-20260427-180842
  Model created.
Creating endpoint config: qwen3vl-neuron-v10-config
  Endpoint config created with InferenceAmiVersion=al2-ami-sagemaker-inference-neuron-2
Creating endpoint: qwen3vl-neuron-v10
  Endpoint creation started.

This will take ~15 minutes (model download + Neuron compilation).
Monitor progress in CloudWatch: /aws/sagemaker/Endpoints/qwen3vl-neuron-v10

In [6]:
# Wait for endpoint to be InService
print(f"Waiting for endpoint {endpoint_name} to be InService...")
print(f"Start time: {time.strftime('%H:%M:%S')}")

while True:
    resp = sm_client.describe_endpoint(EndpointName=endpoint_name)
    status = resp['EndpointStatus']
    print(f"  [{time.strftime('%H:%M:%S')}] Status: {status}")

    if status == 'InService':
        print(f"\nEndpoint is InService!")
        break
    elif status == 'Failed':
        reason = resp.get('FailureReason', 'Unknown')
        print(f"\nEndpoint FAILED: {reason}")
        print(f"Check CloudWatch logs: /aws/sagemaker/Endpoints/{endpoint_name}")
        break
    elif status not in ('Creating', 'Updating'):
        print(f"\nUnexpected status: {status}")
        break

    time.sleep(60)

Waiting for endpoint qwen3vl-neuron-v10 to be InService...
Start time: 01:25:38
  [01:25:38] Status: InService

Endpoint is InService!

## Step 5: Connect to Endpoint

If reconnecting to an existing endpoint (e.g., after a kernel restart), set `endpoint_name` below.

In [7]:
# Reconnect to existing endpoint (uncomment and set endpoint_name if needed)
# endpoint_name = "qwen3vl-neuron-XXXXXXXXXX"
# sm_runtime = boto3.client('sagemaker-runtime', region_name=region)

def invoke_endpoint(payload):
    """Send inference request to endpoint and return parsed response."""
    response = sm_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps(payload)
    )
    return json.loads(response['Body'].read().decode())

print(f"Connected to endpoint: {endpoint_name}")

Connected to endpoint: qwen3vl-neuron-v10

## Step 6: Health Check

In [8]:
# Quick health check
health_payload = {
    "messages": [{"role": "user", "content": "Hello"}],
    "max_tokens": 5
}

t0 = time.time()
response = invoke_endpoint(health_payload)
latency = time.time() - t0

print(f"Status: OK")
print(f"Response: {response['choices'][0]['message']['content']}")
print(f"Latency: {latency:.2f}s")
if 'performance' in response:
    print(f"Server throughput: {response['performance']['tok_per_sec']} tok/s")

Status: OK
Response: Hello! How can I
Latency: 0.63s
Server throughput: 23.1 tok/s

## Step 7: Text-Only Inference

In [9]:
print("=" * 60)
print("TEXT-ONLY INFERENCE")
print("=" * 60)

payload = {
    "messages": [
        {
            "role": "user",
            "content": "Explain what a vision-language model is and how it works. Be concise."
        }
    ],
    "max_tokens": 256
}

t0 = time.time()
response = invoke_endpoint(payload)
latency = time.time() - t0

content = response['choices'][0]['message']['content']
usage = response.get('usage', {})
perf = response.get('performance', {})

print(f"\nResponse ({usage.get('completion_tokens', '?')} tokens, {latency:.2f}s):")
print(content[:2000])
if perf:
    print(f"\nServer-side: {perf.get('tok_per_sec', '?')} tok/s, {perf.get('latency_s', '?')}s")

TEXT-ONLY INFERENCE

Response (107 tokens, 3.38s):
A vision-language model (VLM) is a type of artificial intelligence model that understands and generates text based on visual input (like images) and vice versa. It combines vision and language capabilities, typically using transformers, to map images to text descriptions (e.g., “a cat sitting on a couch”) or generate images from text prompts. VLMs are trained on large datasets of image-text pairs, learning to align visual features with linguistic meaning. Examples include GPT-4V, CLIP, and Flamingo.

Server-side: 32.7 tok/s, 3.276s

## Step 8: Image + Text Inference

Images are sent as base64-encoded data URIs in the OpenAI chat completions format.
The server automatically resizes images to fit the vision encoder's patch budget (max 1024px, 56px aligned).

Supported image formats:
- **Base64 data URI**: `data:image/jpeg;base64,/9j/...` (recommended for reliability)
- **HTTP URL**: `https://example.com/image.jpg` (requires outbound internet from SageMaker)

In [10]:
import urllib.request
from io import BytesIO

def image_url_to_base64(url):
    """Download image from URL and convert to base64 data URI."""
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        img_bytes = resp.read()
    b64 = base64.b64encode(img_bytes).decode('utf-8')
    return f"data:image/jpeg;base64,{b64}"

# Load the Qwen demo image
print("Downloading sample image...")
IMAGE_URL = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
image_b64 = image_url_to_base64(IMAGE_URL)
print(f"Image encoded: {len(image_b64)} chars")

Image encoded: 661883 chars

In [11]:
print("=" * 60)
print("IMAGE + TEXT INFERENCE: Image Description")
print("=" * 60)

payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": image_b64}
                },
                {
                    "type": "text",
                    "text": "Describe this image in detail. What do you see?"
                }
            ]
        }
    ],
    "max_tokens": 256
}

t0 = time.time()
response = invoke_endpoint(payload)
latency = time.time() - t0

content = response['choices'][0]['message']['content']
usage = response.get('usage', {})
perf = response.get('performance', {})

print(f"\nResponse ({usage.get('completion_tokens', '?')} tokens, {latency:.2f}s):")
print(content[:2000])
if perf:
    print(f"\nServer-side: {perf.get('tok_per_sec', '?')} tok/s, {perf.get('latency_s', '?')}s")

IMAGE + TEXT INFERENCE: Image Description

Response (256 tokens, 15.44s):
This is a heartwarming, sun-drenched photograph capturing a tender moment between a woman and her dog on a beach at sunset.

**Central Subjects:**
- A large, light-colored Labrador Retriever is sitting on the sand, facing the woman. The dog is wearing a colorful, patterned harness with a red and blue design. It is extending its front left paw toward the woman in a high-five gesture.
- A young woman with long, dark hair is sitting on the sand, facing the dog. She is wearing a plaid shirt (dark blue and white or black and white) with rolled-up sleeves, and dark pants. She is barefoot. She is smiling warmly, her eyes crinkled with joy as she looks at the dog, and she is holding out her right hand to meet the dog’s paw.

**Setting and Atmosphere:**
- The scene is set on a wide, sandy beach. The sand is textured with footprints and gentle ripples, catching the golden light of the setting sun.
- In the background, the 

In [12]:
print("=" * 60)
print("IMAGE + TEXT INFERENCE: Visual Q&A")
print("=" * 60)

payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": image_b64}
                },
                {
                    "type": "text",
                    "text": "What breed is the dog? What is the woman wearing?"
                }
            ]
        }
    ],
    "max_tokens": 128
}

t0 = time.time()
response = invoke_endpoint(payload)
latency = time.time() - t0

content = response['choices'][0]['message']['content']
usage = response.get('usage', {})
perf = response.get('performance', {})

print(f"\nResponse ({usage.get('completion_tokens', '?')} tokens, {latency:.2f}s):")
print(content[:2000])
if perf:
    print(f"\nServer-side: {perf.get('tok_per_sec', '?')} tok/s, {perf.get('latency_s', '?')}s")

IMAGE + TEXT INFERENCE: Visual Q&A

Response (128 tokens, 8.43s):
Based on the image provided:

**Dog Breed:**
The dog appears to be a **Yellow Labrador Retriever**. This is indicated by its characteristic golden-yellow coat, large, friendly face, and sturdy build. The dog is also wearing a harness, which is common for this breed due to their strong, active nature.

**Woman's Clothing:**
The woman is wearing:
- A **plaid flannel shirt** (or button-down shirt) with a pattern of dark and light squares, likely black and white or dark blue and white.
- **Dark-colored pants** (possibly black or dark denim).
- She is **

Server-side: 18.6 tok/s, 6.889s

## Step 9: Benchmark Suite

Run timed iterations for both text-only and image+text queries.
Reports mean, std, P50, P99 latency and throughput.

In [13]:
import numpy as np

WARMUP_RUNS = 2
BENCHMARK_RUNS = 5

def run_benchmark(payload, warmup=WARMUP_RUNS, runs=BENCHMARK_RUNS, label=""):
    """Run warmup + timed benchmark iterations."""
    print(f"\n{'=' * 60}")
    print(f"BENCHMARK: {label}")
    print(f"  Warmup: {warmup}, Benchmark: {runs}")
    print(f"{'=' * 60}")

    # Warmup
    for i in range(warmup):
        t0 = time.time()
        r = invoke_endpoint(payload)
        lat = time.time() - t0
        toks = r.get('usage', {}).get('completion_tokens', '?')
        server_tps = r.get('performance', {}).get('tok_per_sec', '?')
        print(f"  Warmup {i+1}: {lat:.2f}s, {toks} tokens, {server_tps} tok/s (server)")

    # Timed runs
    latencies = []
    server_throughputs = []
    token_counts = []

    for i in range(runs):
        t0 = time.time()
        r = invoke_endpoint(payload)
        lat = time.time() - t0
        toks = r.get('usage', {}).get('completion_tokens', 0)
        server_tps = r.get('performance', {}).get('tok_per_sec', 0)

        latencies.append(lat)
        server_throughputs.append(server_tps)
        token_counts.append(toks)
        print(f"  Run {i+1}: {lat:.2f}s, {toks} tokens, {server_tps} tok/s")

    lat = np.array(latencies)
    tp = np.array(server_throughputs)

    print(f"\n  --- Results ---")
    print(f"  E2E Latency: {lat.mean():.3f}s (std={lat.std():.3f}, P50={np.percentile(lat, 50):.3f}, P99={np.percentile(lat, 99):.3f})")
    print(f"  Server Throughput: {tp.mean():.1f} tok/s (std={tp.std():.1f}, min={tp.min():.1f}, max={tp.max():.1f})")
    print(f"  Tokens: {np.mean(token_counts):.0f} mean")

    return {
        "label": label,
        "latency_mean": float(lat.mean()),
        "latency_p50": float(np.percentile(lat, 50)),
        "throughput_mean": float(tp.mean()),
    }

print("Benchmark function ready.")

Benchmark function ready.

In [14]:
# Text-only benchmark
text_payload = {
    "messages": [{"role": "user", "content": "Explain what a vision-language model is and how it works. Be concise."}],
    "max_tokens": 256
}

text_results = run_benchmark(text_payload, label="Text-Only")


BENCHMARK: Text-Only
  Warmup: 2, Benchmark: 5
  Warmup 1: 3.37s, 107 tokens, 33.0 tok/s (server)
  Warmup 2: 3.36s, 107 tokens, 33.1 tok/s (server)
  Run 1: 3.37s, 107 tokens, 33.0 tok/s
  Run 2: 3.36s, 107 tokens, 33.0 tok/s
  Run 3: 3.35s, 107 tokens, 33.0 tok/s
  Run 4: 3.39s, 107 tokens, 33.0 tok/s
  Run 5: 3.37s, 107 tokens, 33.0 tok/s

  --- Results ---
  E2E Latency: 3.368s (std=0.014, P50=3.367, P99=3.392)
  Server Throughput: 33.0 tok/s (std=0.0, min=33.0, max=33.0)
  Tokens: 107 mean

In [15]:
# Image+Text benchmark
image_payload = {
    "messages": [{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": image_b64}},
            {"type": "text", "text": "Describe this image in detail. What do you see?"}
        ]
    }],
    "max_tokens": 256
}

image_results = run_benchmark(image_payload, label="Image+Text")


BENCHMARK: Image+Text
  Warmup: 2, Benchmark: 5
  Warmup 1: 14.97s, 256 tokens, 19.2 tok/s (server)
  Warmup 2: 13.98s, 256 tokens, 19.2 tok/s (server)
  Run 1: 13.75s, 256 tokens, 19.2 tok/s
  Run 2: 14.30s, 256 tokens, 19.2 tok/s
  Run 3: 13.63s, 256 tokens, 19.3 tok/s
  Run 4: 14.41s, 256 tokens, 19.2 tok/s
  Run 5: 13.68s, 256 tokens, 19.3 tok/s

  --- Results ---
  E2E Latency: 13.954s (std=0.332, P50=13.745, P99=14.409)
  Server Throughput: 19.2 tok/s (std=0.0, min=19.2, max=19.3)
  Tokens: 256 mean

In [16]:
# Summary
print("\n" + "=" * 60)
print("BENCHMARK SUMMARY")
print("=" * 60)
print(f"\n{'Workload':<15} {'E2E Latency':>12} {'Server tok/s':>14}")
print("-" * 45)
for r in [text_results, image_results]:
    print(f"{r['label']:<15} {r['latency_p50']:>10.2f}s  {r['throughput_mean']:>12.1f}")

print(f"\nEndpoint: {endpoint_name}")
print(f"Instance: {INSTANCE_TYPE}")
print(f"Host AMI: {INFERENCE_AMI_VERSION}")


BENCHMARK SUMMARY

Workload         E2E Latency   Server tok/s
---------------------------------------------
Text-Only             3.37s          33.0
Image+Text           13.75s          19.2

Endpoint: qwen3vl-neuron-v10
Instance: ml.inf2.8xlarge
Host AMI: al2-ami-sagemaker-inference-neuron-2

## Step 10: Cleanup

**Important**: Delete the endpoint when done to stop charges.
Uncomment and run the cell below.

In [17]:
# Uncomment to delete resources:

# sm_client.delete_endpoint(EndpointName=endpoint_name)
# print(f"Endpoint {endpoint_name} deleted")

# sm_client.delete_endpoint_config(EndpointConfigName=config_name)
# print(f"Endpoint config {config_name} deleted")

# sm_client.delete_model(ModelName=model_name)
# print(f"Model {model_name} deleted")

## Technical Notes

### Neuron Host Driver and `InferenceAmiVersion`

SageMaker's default Neuron host AMI ships driver v2.10, which predates Extended ISA (ExtISA) support. The NxD Inference compiler in SDK 2.28 generates NEFFs containing NKI kernel calls (e.g., `cumsum` for on-device sampling) that produce ExtISA instructions. Without the updated AMI, these NEFFs fail to load with:

```
tdrv_get_device_resource_va failed (ret=4) to get event semaphore vaddr
RuntimeError: Could not load the model status=4 message=Allocation Failure
```

The `InferenceAmiVersion` parameter in `ProductionVariant` selects an alternate host AMI:

| Value | Accelerator | Driver |
|-------|------------|--------|
| `al2-ami-sagemaker-inference-neuron-2` | Inferentia2 / Trainium | Neuron 2.19 |
| `al2-ami-sagemaker-inference-gpu-2` | GPU | NVIDIA 535 / CUDA 12.2 |
| `al2-ami-sagemaker-inference-gpu-3-1` | GPU | NVIDIA 550 / CUDA 12.4 |
| `al2023-ami-sagemaker-inference-gpu-4-1` | GPU | NVIDIA 580 / CUDA 13.0 |

**This parameter requires boto3 >= 1.42** (or AWS CLI > 2.16). The SageMaker Python SDK's `Model.deploy()` does not currently expose this parameter, which is why this notebook uses boto3 directly.

### Container Compilation

The model compiles from scratch on the SageMaker instance (~10 minutes on inf2.8xlarge). The compilation produces NEFFs compatible with the host driver. The compile cache is cleared at container startup to ensure fresh compilation.

To reduce startup time on subsequent deployments, you could save the compiled artifacts to S3 and include them in the model data. However, the artifacts are tied to the specific host driver version, so they may not be portable across different `InferenceAmiVersion` values.

### inf2 Limitations

- **ISA kernels not supported** (compiler issue NCC_IBVF032) -- all kernel flags must be off
- **LNC=1 only** (NeuronCore v2 does not support LNC=2)
- **tp=1 NOT viable** on SDK 2.28 (compiler ICE)
- **Slower than trn2** -- inf2.8xlarge ~30 tok/s text vs trn2 ~95-117 tok/s

### SDK 2.28 Patches Applied in Container

1. **tie_word_embeddings**: Monkey-patches `NeuronQwen3VLForCausalLM.update_state_dict_for_tied_weights` to copy `embed_tokens.weight` -> `lm_head.weight` for the 4B model
2. **vllm_neuron Sampler import**: Fixes `TypeError: 'module' object is not callable` when `NEURON_ON_DEVICE_SAMPLING_DISABLED=1` is set
3. **Writable model overlay**: Creates `/tmp/model` with symlinks to read-only `/opt/ml/model` so NxDI can write `neuron-compiled-artifacts/`